In [5]:
import re
import numpy as np
import pandas as pd
CSV_PATH = "cleaned_reviews.csv"
TEXT_COL = "리뷰내용"

df = pd.read_csv(CSV_PATH)
print(f"로드 완료: {len(df):,}건")

# 리뷰 본문 NaN 제거
before = len(df)
df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
print(f"NaN 제거: {before:,} → {len(df):,}건 (-{before-len(df):,})")

C:\Users\Minsoo\AppData\Local\Temp\ipykernel_10128\1664386471.py:7: DtypeWarning: Columns (0: 만족도, 1: 구매사이즈, 2: 구매상세, 3: 사이즈, 4: 화면 대비 색감, 5: 퀄리티, 6: 구김, 7: 두께감, 8: 신축성, 9: 색감, 10: 보온성) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


로드 완료: 685,292건
NaN 제거: 685,292 → 685,042건 (-250)


In [6]:
# 노이즈 확인하기
sample = df[TEXT_COL].astype(str).astype('object')

print("[노이즈 패턴 분포]")
print(f"엑셀 오류값(#NAME? 등): {sample.str.contains(r'#(NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)', regex=True, na=False).sum():,}건")
print(f"HTML 태그(<br> 등): {sample.str.contains(r'<[^>]+>', regex=True, na=False).sum():,}건")
print(f"HTML 엔티티(&nbsp;): {sample.str.contains(r'&[a-z]+;', regex=True, na=False).sum():,}건")
print(f"URL: {sample.str.contains(r'https?://|www\.', regex=True, na=False).sum():,}건")
print(f"줄바꿈/탭: {sample.str.contains(r'[\r\n\t]', regex=True, na=False).sum():,}건")
print(f"반복문자 4회+: {sample.str.contains(r'(.)\1{3,}', regex=True, na=False).sum():,}건")
print(f"한글 없음: {(~sample.str.contains(r'[가-힣]', regex=True, na=False)).sum():,}건")

no_korean = df[~sample.str.contains(r'[가-힣]', regex=True, na=False)]
print(f"\n[한글 없는 리뷰 샘플]")
display(no_korean[TEXT_COL].head(10))

[노이즈 패턴 분포]


C:\Users\Minsoo\AppData\Local\Temp\ipykernel_10128\2729053620.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print(f"엑셀 오류값(#NAME? 등): {sample.str.contains(r'#(NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)', regex=True, na=False).sum():,}건")


엑셀 오류값(#NAME? 등): 0건
HTML 태그(<br> 등): 234건
HTML 엔티티(&nbsp;): 2,167건
URL: 5건
줄바꿈/탭: 0건


C:\Users\Minsoo\AppData\Local\Temp\ipykernel_10128\2729053620.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print(f"반복문자 4회+: {sample.str.contains(r'(.)\1{3,}', regex=True, na=False).sum():,}건")


반복문자 4회+: 5,994건
한글 없음: 184건

[한글 없는 리뷰 샘플]


14184                    采用优质棉质面料，吸汗透气舒适，大小合适，买到合体的衣服，非常高兴！
31198                                whehudisjwndnckcohxhsn
37415     GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD
43875                                  14163662626277262671
49392                                 goo1oo1oo1o1oo1o1ood!
54024                               12315152515261625616125
75028                                  Djkfruogvmxsdkiroogh
95592     &Aacute;o sấu v&agrave; t&ocirc;i mua về d&ugr...
97058                            this pants very recomended
102063                           this pants very recomended
Name: 리뷰내용, dtype: str

- 한글 없음 : 184건 모두 한국어 BERTopic 분석에서 어차피 제외되어야 함. 셀 4의 is_valid_for_topic 필터에서 자동으로 걸러짐 (한글 5자 이상)

In [7]:
from tqdm.auto import tqdm

# 통합 정제 함수
def clean_review_text(text):
    """
    무신사 리뷰 텍스트 1단계 전처리.
    임베딩(ko-sroberta) 입력용 정제 텍스트를 반환한다.
    """
    text = str(text)

    # (1) 엑셀 오류값 제거 (-> 0건)
    text = re.sub(r'#(NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)[!?]?', ' ', text)

    # (2) URL 제거 -> (5건)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # (3) HTML 태그 + 엔티티 제거 -> (234건 + 2,167건)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)

    # (4) 줄바꿈·탭 → 공백 -> (0건)
    text = re.sub(r'[\r\n\t]+', ' ', text)

    # (5) 반복문자 정규화
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1{2,}', r'\1\1', text) #같은 자음모음이 총 3번 이상 나오면 2번으로 줄임
    text = re.sub(r'(.)\1{3,}', r'\1\1', text) #한글(완성형), 영어, 숫자, 기호 등 아무 문자가 총 3번 이상 나오면 2번으로 줄임

    # (6) 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()

    return text

tqdm.pandas()
df['리뷰내용_clean'] = df[TEXT_COL].progress_apply(clean_review_text)
print("정제 완료")

  0%|          | 0/685042 [00:00<?, ?it/s]

정제 완료


- (5) 반복문자 정규화 한계 : 패턴반복(ex.와우와우와우)은 처리 안함. 추후 확인 후에 진행.

In [9]:
# 유효리뷰 플래그
df['한글_글자수'] = df['리뷰내용_clean'].str.count(r'[가-힣]')
df['전체_글자수'] = df['리뷰내용_clean'].str.len()

df['is_valid_for_topic'] = df['한글_글자수'] >= 5

print(f"[유효 리뷰 분포]")
print(f"  유효 (한글 5자 이상): {df['is_valid_for_topic'].sum():,}건 "
      f"({df['is_valid_for_topic'].mean()*100:.2f}%)")
print(f"  무효 (한글 5자 미만): {(~df['is_valid_for_topic']).sum():,}건")

print("\n[무효로 분류된 리뷰 샘플]")
display(df.loc[~df['is_valid_for_topic'], ['리뷰내용', '리뷰내용_clean', '한글_글자수']].head(10))

[유효 리뷰 분포]
  유효 (한글 5자 이상): 684,745건 (99.96%)
  무효 (한글 5자 미만): 297건

[무효로 분류된 리뷰 샘플]


,리뷰내용,리뷰내용_clean,한글_글자수
5451,어어어어어어어어어어어어어어어어어어어엉,어어엉,3
14184,采用优质棉质面料，吸汗透气舒适，大小合适，买到合体的衣服，非常高兴！,采用优质棉质面料，吸汗透气舒适，大小合适，买到合体的衣服，非常高兴！,0
15368,솦호호56)?56&))₩₩(6)((5)(tygfyhdryhgfh,솦호호56)?56&))₩₩(6)((5)(tygfyhdryhgfh,3
16593,굳,굳,1
21129,굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿,굿굿,2
31198,whehudisjwndnckcohxhsn,whehudisjwndnckcohxhsn,0
35450,12345567889909765543호ㅓㅗ포ㅜ,12345567889909765543호ㅓㅗ포ㅜ,2
37415,GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD,GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD,0
43875,14163662626277262671,14163662626277262671,0
49392,goo1oo1oo1o1oo1o1ood!,goo1oo1oo1o1oo1o1ood!,0


In [10]:
# 전후 비교검증
print("[정제 전후 비교 샘플]")
sample_idx = df[df['리뷰내용'] != df['리뷰내용_clean']].sample(10, random_state=42).index
for i in sample_idx:
    print(f"\n리뷰번호: {df.loc[i, '리뷰번호']}")
    print(f"  Before: {repr(df.loc[i, '리뷰내용'])[:120]}")
    print(f"  After : {repr(df.loc[i, '리뷰내용_clean'])[:120]}")

print("\n[글자 수 통계]")
print(df[['전체_글자수', '한글_글자수']].describe())

changed = (df['리뷰내용'].astype(str) != df['리뷰내용_clean']).sum()
print(f"\n정제로 인해 변경된 리뷰: {changed:,}건 ({changed/len(df)*100:.1f}%)")

[정제 전후 비교 샘플]

리뷰번호: 31385597
  Before: '약간 오버핏이기는 한데 반바지랑 아주 잘어울려요 이쁩니다 '
  After : '약간 오버핏이기는 한데 반바지랑 아주 잘어울려요 이쁩니다'

리뷰번호: 63322193
  Before: '건조기 돌릴 것 까지 생각해서 크게 사요 사이즈 표기 그대로입니다 좋아요 '
  After : '건조기 돌릴 것 까지 생각해서 크게 사요 사이즈 표기 그대로입니다 좋아요'

리뷰번호: 20267850
  Before: '청바지에 입기 좋고  오버해서 앞넣입 너무 좋네요'
  After : '청바지에 입기 좋고 오버해서 앞넣입 너무 좋네요'

리뷰번호: 55885011
  Before: '구매한지 얼마 안됐을 때는 굴러갈 것 같았는데, 지금은  솜이 좀 죽어서 그런가 괜찮은 것 같아요. '
  After : '구매한지 얼마 안됐을 때는 굴러갈 것 같았는데, 지금은 솜이 좀 죽어서 그런가 괜찮은 것 같아요.'

리뷰번호: 9850732
  Before: '다리가 굵고  짧아서   온라인으로 바지를 사면  거의 반품했어야 했는데  이 바지는  그냥  아주 딱 맞습니다  . 다음에도  또 바지는 제멋에서 살려고합니다'
  After : '다리가 굵고 짧아서 온라인으로 바지를 사면 거의 반품했어야 했는데 이 바지는 그냥 아주 딱 맞습니다 . 다음에도 또 바지는 제멋에서 살려고합니다'

리뷰번호: 6783071
  Before: '색상별로 사봤는데 색은 이쁘게 잘나옴  가성비는 진짜좋음'
  After : '색상별로 사봤는데 색은 이쁘게 잘나옴 가성비는 진짜좋음'

리뷰번호: 54424251
  Before: '무난한 오버핏 원했는데 딱 적당하고 재질도 좋네요 '
  After : '무난한 오버핏 원했는데 딱 적당하고 재질도 좋네요'

리뷰번호: 72747670
  Before: '배송도 생각보다  많이 빠르고  디자인도 핏도 맘에들어요'
  After : '배송도 생각보다 많이 빠르고 디자인도 핏도 맘에들어요'


- min = 0(완전히 빈 리뷰)을 확인 할 수 있음. 해당 리뷰 확인필요

In [11]:
# '글자 수 통계'의 min이 0인값 확인
empty_after = df[df['전체_글자수'] == 0]
print(f"정제 후 빈 리뷰: {len(empty_after):,}건")
print("\n[원본 확인]")
display(empty_after[['리뷰번호', '리뷰내용']].head(20))

정제 후 빈 리뷰: 1건

[원본 확인]


,리뷰번호,리뷰내용
279991,24482388,https://youtu.be/8ebPs7hKrc4


- 리뷰 내용 원본이 url 코드로 되어있음.
- 정제 후 : 빈 문자열
- 한글 글자수 = 0 이라 is_valid_for_topic = False로 자동 분류됨.
- 셀 2의 URL 포함 리뷰는 5건 -> 나머지 4건은 URL + 본문 텍스트로 예상됨.

In [14]:
df_topic = df[df['is_valid_for_topic']].reset_index(drop=True)
docs = df_topic['리뷰내용_clean'].tolist()

print(f"BERTopic 입력 문서 수: {len(docs):,}건")
print(f"전체 대비 유효 비율: {len(docs)/len(df)*100:.2f}%")

output_path = f"./reviews_step1_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"\n저장 완료: {output_path}")

BERTopic 입력 문서 수: 684,745건
전체 대비 유효 비율: 99.96%
